In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance

In [2]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [3]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers=2, num_layers_sleep=2):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(hidden_wake_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw

In [4]:
class compressor(nn.Module):
    def __init__(self, input_size, hidden_compressor_size, num_layers=1):
        super(compressor, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_compressor_size, num_layers, nonlinearity='relu', batch_first=True)
        self.compressor_fc = nn.Linear(hidden_compressor_size, 2)

    def forward(self, x, hc=None):
        if hc == None:
            out, hc = self.rnn(x)
        else:
            out, hc = self.rnn(x, hc)

        out = self.compressor_fc(out)
        
        return out, hc

In [5]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [6]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [7]:
class Dataset_converter_compressor(Dataset):
    def __init__(self, data, mask):
        total_sample = len(data)
        self.X = np.zeros((total_sample-2, len(tokens)))
        self.y = np.zeros((total_sample-2, 2))
        for ii in range(total_sample-2):
            token = data[ii]
            self.X[ii, ord(token)-65] = 1 
            self.y[ii,mask[ii]] = 1
            

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [173]:
### initial training ###
total_samples = 20000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 35
hidden_sleep_size = 35
sleep_output_size = 20
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, hw=mem)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 2.1401, accuracy: 0.2460
Iter : 2001, loss: 2.0534, accuracy: 0.2540
Iter : 3001, loss: 1.6411, accuracy: 0.3830
Iter : 4001, loss: 2.3275, accuracy: 0.5520
Iter : 5001, loss: 1.8585, accuracy: 0.5910
Iter : 6001, loss: 2.0520, accuracy: 0.6560
Iter : 7001, loss: 1.3971, accuracy: 0.6810
Iter : 8001, loss: 2.2266, accuracy: 0.6760
Iter : 9001, loss: 1.9910, accuracy: 0.6840
Iter : 10001, loss: 1.9061, accuracy: 0.6630
Iter : 11001, loss: 1.6352, accuracy: 0.6700
Iter : 12001, loss: 1.8524, accuracy: 0.6530
Iter : 13001, loss: 2.9716, accuracy: 0.6550
Iter : 14001, loss: 2.7002, accuracy: 0.6460
Iter : 15001, loss: 1.5628, accuracy: 0.6640
Iter : 16001, loss: 2.0190, accuracy: 0.6550
Iter : 17001, loss: 1.1729, accuracy: 0.6780
Iter : 18001, loss: 1.7147, accuracy: 0.6580
Iter : 19001, loss: 1.5161, accuracy: 0.6830


In [174]:
centroids = []

threshold = .6
n_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0

for ii in range(n_samples):
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X_hat.reshape(1,1,-1))
        centroids.append(mem[0][0])
    else:
        X_hat, mem = network1(X_hat, mem)

    dis = []
    min_dis = 10
    min_dis_id = -1
    for jj in range(len(centroids)):
        dis.append(
            distance.cosine(centroids[jj].detach().numpy(), mem[0][0].detach().numpy())
        )
        if min_dis >= dis[-1]:
            min_dis = dis[-1] 
            min_dis_id = jj 
    if min_dis < threshold:
        centroids[min_dis_id] = (centroids[min_dis_id] + mem[0][0])/2.0
    else:
        centroids.append(mem[0][0])

    # print(min_dis)   
    X_hat = torch.nn.functional.softmax(X_hat, dim=1)
    dist_categ = torch.distributions.Categorical(probs=X_hat.reshape(-1))
    idx = dist_categ.sample()

    X_hat = torch.zeros(len(tokens),dtype=torch.float32)
    X_hat[idx] = 1.0
    X_hat = X_hat.reshape(1,1,-1)   
    

In [175]:
centroids

[tensor([9.5916e-01, 1.9744e-01, 1.5622e-01, 7.8939e-01, 2.8189e-01, 6.0687e-01,
         1.7544e-02, 8.6559e-03, 6.2462e-01, 8.8604e-03, 1.9132e-01, 2.8483e-01,
         1.1487e+00, 0.0000e+00, 3.4807e-01, 4.6102e-01, 0.0000e+00, 7.8644e-01,
         0.0000e+00, 4.2990e-03, 1.4417e-01, 4.7149e-01, 0.0000e+00, 9.9397e-02,
         3.5367e-04, 3.1471e-02, 1.7646e-01, 0.0000e+00, 6.7496e-01, 0.0000e+00,
         7.8705e-03, 2.0632e-01, 4.5719e-01, 2.2251e-01, 2.2371e-01],
        grad_fn=<DivBackward0>),
 tensor([0.0000, 1.1176, 0.2783, 0.4430, 0.0193, 0.0000, 0.0680, 0.3610, 0.7773,
         0.0000, 0.0000, 0.1305, 0.0000, 0.0000, 0.6249, 0.0158, 0.5432, 0.0000,
         0.0000, 0.6038, 0.0201, 0.0000, 0.0000, 0.0000, 0.0532, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4019, 0.0000, 0.6006],
        grad_fn=<DivBackward0>),
 tensor([0.0000, 0.0291, 0.3409, 0.7607, 0.2061, 0.3724, 0.6226, 0.3698, 0.0000,
         0.0000, 0.5474, 0.0000, 0.1074, 0.0000, 0.3593, 0.00

In [176]:
n_samples = 1000
data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

community = {}
for ii in range(len(centroids)):
    community[ii] = []

ii = 0

for X, y in train_loader:
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X.reshape(1,1,-1))
        # centroids.append(mem[0][0])
    else:
        X_hat, mem = network1(X, mem)
    
    dis = []
    for center in centroids:
        dis.append(
            distance.cosine(center.detach().numpy(), mem[0][0].detach().numpy())
        )
    
    idx = np.argmin(dis)
    community[idx].append(tokens[X.argmax(axis=2)])

         
    ii += 1

In [177]:
for com in community.keys():
    print(np.unique(community[com]))

['A' 'B' 'C']
['G']
['D' 'E' 'F']


In [90]:
sleep_samples = 40
compressed_seq = ''
data_sleep = get_sequence(sleep_samples, n_community, n_members, train_percent=1.0)
data_set_sleep = Dataset_converter(data_sleep, working_memory, short_term_memory)

sleep_loader = DataLoader(data_set_sleep, batch_size=1, shuffle=False)

network1.rnn.requires_grad = True
network1.wake_fc.requires_grad = True

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
hidden_s = None
correct = np.zeros(1000,dtype=float)
communities = [0]
current_community = 0
# changed_com = False
for X, y in sleep_loader:
    # print( tokens[X.argmax(axis=2)], current_community)
    optimizer.zero_grad()
    # print(data_sleep[total])
    if total == 0:
        predicted_y, hidden_w = network1(X)
    elif total==1:
        predicted_y, hidden_w, hidden_s = network1(X, community, hw=mem, sleep=True)
    else:
        predicted_y, hidden_w, hidden_s = network1(X, community, hw=mem, hs=mem_, sleep=True)
    
    # print(predicted_y.shape)
    loss = criterion(predicted_y, y)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        mem=hidden_w.clone()

        if total != 0:
            mem_=hidden_s.clone()

        if distance.cosine(centroids[0].detach().numpy(), mem[0][0].detach().numpy()) < distance.cosine(centroids[1].detach().numpy(), mem[0][0].detach().numpy()):
             if communities[-1] != 0:
                  communities.append(0)
                  current_community = communities.pop(0)
                  community = centroids[current_community].view(1,1,-1)

             print('community 0', tokens[X.argmax(axis=2)], communities)
        else:
            #  print('yes', communities[-1])
             if communities[-1] != 1:
                  communities.append(1)
                  current_community = communities.pop(0)
                  community = centroids[current_community].view(1,1,-1)
                #   print('eije cuurent', current_community, communities)

             print('community 1', tokens[X.argmax(axis=2)])

        # if total == 0:
        #      community = current_community
        #      prev_community = current_community
        #      prev_community_no = current_community_no
        # else:
        #      if current_community_no != prev_community_no:
        #           community = prev_community
        #           prev_community = current_community
        #           prev_community_no = current_community_no


        true_y = y.argmax(axis=1)
        # print(predicted_y)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

community 1 C
community 1 A
community 1 B
community 1 G
community 0 D [0]
community 0 E [0]
community 0 F [0]
community 1 G
community 0 D [0]
community 0 E [0]
community 0 F [0]
community 1 G
community 1 A
community 1 C
community 1 B
community 1 G
community 0 F [0]
community 0 D [0]
community 0 E [0]
community 1 G
community 0 E [0]
community 0 F [0]
community 0 D [0]
community 1 G
community 0 F [0]
community 0 E [0]
community 0 D [0]
community 1 G
community 0 F [0]
community 0 E [0]
community 0 D [0]
community 1 G
community 0 D [0]
community 0 F [0]
community 0 E [0]
community 1 G
community 0 E [0]
community 0 D [0]
